# Multimodal RAG System — Full Pipeline
### Groq · Qdrant · Cloudinary · CLIP · Whisper

**What this notebook does end-to-end:**
```
Upload file (text / image / audio / video)
          ↓
  Cloudinary  ←─ stores original file, returns public URL
          ↓
  Preprocessing
    • text   → chunk
    • image  → CLIP embed + zero-shot tag suggestion
    • audio  → Groq Whisper transcription → chunk
    • video  → extract audio (Whisper) + sample frames (CLIP)
          ↓
  Embeddings
    • text chunks  → sentence-transformers (384-dim)
    • image frames → CLIP (512-dim)
          ↓
  Qdrant (two collections: rag_text / rag_images)
          ↓
  Query → embed → Qdrant similarity search → top-k context
          ↓
  Groq LLaMA-3 → grounded answer + cited sources
```


## Step 1 — Install Dependencies

In [ ]:
%%capture
!pip install groq 'qdrant-client>=1.7.0' sentence-transformers
!pip install torch torchvision transformers pillow
!pip install moviepy opencv-python-headless pydub
!pip install cloudinary numpy requests tqdm
!apt-get install -y ffmpeg > /dev/null 2>&1
print('All dependencies installed!')

## Step 2 — API Keys & Configuration

In [ ]:
import os

# API keys and secrets are redacted. Set them via environment variables before running.
import os
GROQ_API_KEY          = os.environ.get("GROQ_API_KEY", "REDACTED")
QDRANT_URL            = os.environ.get("QDRANT_URL", "")
QDRANT_API_KEY        = os.environ.get("QDRANT_API_KEY", "REDACTED")

CLOUDINARY_CLOUD_NAME = os.environ.get("CLOUDINARY_CLOUD_NAME", "")
CLOUDINARY_API_KEY    = os.environ.get("CLOUDINARY_API_KEY", "REDACTED")
CLOUDINARY_API_SECRET = os.environ.get("CLOUDINARY_API_SECRET", "REDACTED")

TEXT_COLLECTION  = "rag_text"
IMAGE_COLLECTION = "rag_images"

GROQ_LLM_MODEL = "llama-3.3-70b-versatile"
GROQ_ASR_MODEL = "whisper-large-v3"
EMBED_MODEL    = "all-MiniLM-L6-v2"
CLIP_MODEL_ID  = "openai/clip-vit-base-patch32"

TOP_K         = 5
CHUNK_SIZE    = 300
CHUNK_OVERLAP = 50

os.environ["GROQ_API_KEY"] = GROQ_API_KEY
print("Configuration saved.")

## Step 3 — Imports & Client Initialisation

In [ ]:
import os, uuid, textwrap, tempfile
from pathlib import Path
from typing import List, Dict, Optional

import numpy as np
import torch
from PIL import Image
from tqdm import tqdm

# Groq
from groq import Groq
groq_client = Groq(api_key=GROQ_API_KEY)

# Qdrant
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
qdrant_client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)

# Cloudinary
import cloudinary
import cloudinary.uploader
cloudinary.config(
    cloud_name = CLOUDINARY_CLOUD_NAME,
    api_key    = CLOUDINARY_API_KEY,
    api_secret = CLOUDINARY_API_SECRET,
    secure     = True,
)

# Text Embedder
print("Loading sentence-transformers …")
from sentence_transformers import SentenceTransformer
text_embedder = SentenceTransformer(EMBED_MODEL)

TEXT_DIM = text_embedder.get_sentence_embedding_dimension()

# CLIP
print("Loading CLIP …")
from transformers import CLIPProcessor, CLIPModel
clip_model     = CLIPModel.from_pretrained(CLIP_MODEL_ID)
clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_ID)
CLIP_DIM       = clip_model.config.projection_dim   # 512

print(f"\nAll clients ready")
print(f"   text_dim = {TEXT_DIM}  |  clip_dim = {CLIP_DIM}")
print(f"   Qdrant connected: {qdrant_client.get_collections() is not None}")

## Step 4 — Qdrant Collections Setup

In [ ]:
def ensure_collection(name: str, dim: int, distance=Distance.COSINE):
    """Create a Qdrant collection if it doesn't already exist."""
    existing = {c.name for c in qdrant_client.get_collections().collections}
    if name not in existing:
        qdrant_client.create_collection(
            collection_name=name,
            vectors_config=VectorParams(size=dim, distance=distance),
        )
        print(f"  Created  '{name}'  (dim={dim})")
    else:
        print(f"  '{name}' already exists")

print("Setting up Qdrant collections …")
ensure_collection(TEXT_COLLECTION,  TEXT_DIM)
ensure_collection(IMAGE_COLLECTION, CLIP_DIM)
print("\nCollections ready.")

## Step 5 — Cloudinary Upload Helper

In [ ]:
def upload_to_cloudinary(local_path: str, resource_type: str = "auto") -> Dict:
    """
    Upload a local file to Cloudinary.
    Returns dict with 'url' (CDN URL) and 'public_id' (for deletion).
    resource_type: 'image' | 'video' | 'raw' | 'auto'
    """
    result = cloudinary.uploader.upload(
        local_path,
        resource_type=resource_type,
        folder="multimodal_rag",
    )
    return {
        "url":       result["secure_url"],
        "public_id": result["public_id"],
    }

def delete_from_cloudinary(public_id: str, resource_type: str = "image"):
    """Remove a file from Cloudinary by its public_id."""
    cloudinary.uploader.destroy(public_id, resource_type=resource_type)

print("Cloudinary helpers ready.")

## Step 6 — Text Chunking & Embedding

In [ ]:
def chunk_text(text: str, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP) -> List[str]:
    """Split text into overlapping word-count chunks."""
    words = text.split()
    if not words:
        return []
    step   = max(1, chunk_size - overlap)
    chunks = []
    for i in range(0, len(words), step):
        chunk = " ".join(words[i : i + chunk_size])
        if chunk.strip():
            chunks.append(chunk)
        if i + chunk_size >= len(words):
            break
    return chunks or [text]

def embed_texts(texts: List[str]) -> np.ndarray:
    """Return L2-normalised sentence embeddings as numpy array."""
    return text_embedder.encode(
        texts,
        show_progress_bar=False,
        normalize_embeddings=True,
    )

## Step 7 — CLIP Image Embedding & Zero-Shot Tagging

In [ ]:
TAG_CANDIDATES = [
    "person", "people", "crowd",
    "animal", "dog", "cat", "bird", "wildlife",
    "nature", "forest", "mountain", "beach", "ocean", "sky", "sunset",
    "city", "street", "building", "architecture", "interior",
    "food", "drink", "meal", "restaurant",
    "vehicle", "car", "airplane", "boat",
    "sport", "outdoor activity", "exercise",
    "technology", "computer", "phone", "screen",
    "art", "painting", "drawing", "abstract",
    "document", "text", "diagram", "chart", "graph",
    "medical", "science", "laboratory",
    "fashion", "clothing", "accessories",
    "event", "celebration", "wedding", "conference",
    "landscape", "rural", "urban", "aerial view",
]

def _to_tensor(output) -> torch.Tensor:
    """
    FIX: newer transformers return BaseModelOutputWithPooling, not a plain
    tensor.  Extract the actual float tensor regardless of version.
    """
    if isinstance(output, torch.Tensor):
        return output
    if hasattr(output, "pooler_output") and output.pooler_output is not None:
        return output.pooler_output
    if hasattr(output, "last_hidden_state"):
        return output.last_hidden_state[:, 0, :]
    raise ValueError(f"Cannot extract tensor from type: {type(output)}")

def embed_image(image: Image.Image) -> np.ndarray:
    """L2-normalised CLIP image embedding."""
    inputs = clip_processor(images=image, return_tensors="pt")
    with torch.no_grad():
        raw = clip_model.get_image_features(**inputs)
    feats = _to_tensor(raw)
    feats = feats / feats.norm(dim=-1, keepdim=True)
    return feats.squeeze().cpu().numpy()

def suggest_tags(image: Image.Image, top_k: int = 8, threshold: float = 0.18) -> List[str]:
    """Zero-shot image tagging via CLIP text-image cosine similarity."""
    text_inputs = clip_processor(
        text=[f"a photo of {t}" for t in TAG_CANDIDATES],
        return_tensors="pt", padding=True, truncation=True,
    )
    img_inputs = clip_processor(images=image, return_tensors="pt")

    with torch.no_grad():
        raw_text = clip_model.get_text_features(**text_inputs)
        raw_img  = clip_model.get_image_features(**img_inputs)

    text_feats = _to_tensor(raw_text)
    img_feats  = _to_tensor(raw_img)

    text_feats = text_feats / text_feats.norm(dim=-1, keepdim=True)
    img_feats  = img_feats  / img_feats.norm(dim=-1, keepdim=True)

    scores = (img_feats @ text_feats.T).squeeze().tolist()
    if isinstance(scores, float):
        scores = [scores]
    ranked = sorted(zip(TAG_CANDIDATES, scores), key=lambda x: x[1], reverse=True)
    return [tag for tag, score in ranked[:top_k] if score >= threshold]

## Step 8 — Audio Transcription via Groq Whisper

In [ ]:
def transcribe_audio(file_path: str) -> str:
    """
    Transcribe a local audio/video file using Groq Whisper.
    Handles files > 25 MB by splitting into 10-min segments with pydub.
    Supported formats: mp3, mp4, mpeg, mpga, m4a, wav, webm.
    """
    MAX_BYTES = 24 * 1024 * 1024

    def _transcribe_single(path: str) -> str:
        with open(path, "rb") as f:
            result = groq_client.audio.transcriptions.create(
                file=(Path(path).name, f),
                model=GROQ_ASR_MODEL,
                response_format="text",
            )

        return result if isinstance(result, str) else result.text

    if os.path.getsize(file_path) <= MAX_BYTES:
        return _transcribe_single(file_path)

    from pydub import AudioSegment
    audio    = AudioSegment.from_file(file_path)
    seg_ms   = 10 * 60 * 1000
    segments = [audio[i : i + seg_ms] for i in range(0, len(audio), seg_ms)]
    texts    = []
    with tempfile.TemporaryDirectory() as tmp:
        for idx, seg in enumerate(tqdm(segments, desc="Transcribing segments")):
            seg_path = os.path.join(tmp, f"seg_{idx}.mp3")
            seg.export(seg_path, format="mp3")
            texts.append(_transcribe_single(seg_path))
    return " ".join(texts)

## Step 9 — Video Processing

In [ ]:
def extract_video_frames(video_path: str, max_frames: int = 12) -> List[Image.Image]:
    """Uniformly sample up to max_frames PIL Images from a video file."""
    import cv2
    cap   = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    step  = max(1, total // max_frames)
    frames: List[Image.Image] = []
    for i in range(0, total, step):
        cap.set(cv2.CAP_PROP_POS_FRAMES, i)
        ret, frame = cap.read()
        if ret:
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(Image.fromarray(rgb))
        if len(frames) >= max_frames:
            break
    cap.release()
    return frames

def extract_audio_from_video(video_path: str, out_dir: str) -> Optional[str]:
    """Extract audio track from a video and save as mp3. Returns path or None."""
    from moviepy.editor import VideoFileClip
    audio_path = os.path.join(out_dir, "audio.mp3")

    clip = VideoFileClip(video_path)
    try:
        if clip.audio is None:
            return None
        clip.audio.write_audiofile(audio_path, verbose=False, logger=None)
        return audio_path
    finally:
        clip.close()

## Step 10 — Qdrant Upsert Helpers

In [ ]:
def upsert_text_chunks(
    chunks:   List[str],
    source:   str,
    modality: str,
    cloudinary_url: str = "",
):
    """Embed text chunks and upsert into rag_text collection."""
    if not chunks:
        return
    embeddings = embed_texts(chunks)
    points = [
        PointStruct(
            id=str(uuid.uuid4()),
            vector=emb.tolist(),
            payload={
                "text":           ch,
                "source":         source,
                "modality":       modality,
                "cloudinary_url": cloudinary_url,
            },
        )
        for ch, emb in zip(chunks, embeddings)
    ]
    qdrant_client.upsert(collection_name=TEXT_COLLECTION, points=points)
    print(f"  {len(points)} text chunks [{modality}] → Qdrant")


def upsert_image_vector(
    image:         Image.Image,
    source:        str,
    tags:          List[str],
    cloudinary_url: str = "",
):
    """Embed image with CLIP and upsert into rag_images collection."""
    emb = embed_image(image)
    point = PointStruct(
        id=str(uuid.uuid4()),
        vector=emb.tolist(),
        payload={
            "source":         source,
            "tags":           tags,
            "caption":        "Image containing: " + ", ".join(tags),
            "modality":       "image",
            "cloudinary_url": cloudinary_url,
        },
    )
    qdrant_client.upsert(collection_name=IMAGE_COLLECTION, points=[point])

## Step 11 — Ingestion Pipeline (text / image / audio / video)

In [ ]:
# TEXT
def ingest_text(text: str, source_name: str = "user_text"):
    """
    Ingest raw text:
      1. Chunk
      2. Embed → Qdrant
    No Cloudinary upload (text is stored inline in payloads).
    """
    print(f"Ingesting text '{source_name}' ({len(text)} chars)")
    chunks = chunk_text(text)
    upsert_text_chunks(chunks, source=source_name, modality="text")
    print(f"   Done — {len(chunks)} chunk(s)")


# IMAGE
def ingest_image(image_path: str) -> List[str]:
    """
    Ingest a local image:
      1. Upload to Cloudinary (persistent CDN storage)
      2. CLIP embed + zero-shot tag suggestion
      3. Store image vector + tag-description text → Qdrant
    Returns suggested tags.
    """
    print(f"Ingesting image '{image_path}'")

    #Cloudinary
    cld = upload_to_cloudinary(image_path, resource_type="image")
    print(f"   Cloudinary URL: {cld['url']}")

    #CLIP
    img  = Image.open(image_path).convert("RGB")
    tags = suggest_tags(img)
    print(f"   Tags: {tags}")

    #Image vector → rag_images
    upsert_image_vector(img, source=cld["url"], tags=tags, cloudinary_url=cld["url"])

    #Tag text → rag_text
    tag_text = f"Image: {Path(image_path).name}. Tags: {', '.join(tags)}. Cloudinary: {cld['url']}"
    upsert_text_chunks([tag_text], source=cld["url"], modality="image", cloudinary_url=cld["url"])

    return tags


# AUDIO
def ingest_audio(audio_path: str) -> str:
    """
    Ingest a local audio file:
      1. Upload to Cloudinary (raw resource)
      2. Groq Whisper transcription
      3. Chunk transcript → Qdrant
    Returns transcript.
    """
    print(f"Ingesting audio '{audio_path}'")

    # 1. Cloudinary
    cld = upload_to_cloudinary(audio_path, resource_type="raw")
    print(f"   Cloudinary URL: {cld['url']}")

    # 2. Transcribe
    transcript = transcribe_audio(audio_path)
    print(f"   Transcript preview: {transcript[:120]}...")

    # 3. Chunk + embed → Qdrant
    chunks = chunk_text(transcript)
    upsert_text_chunks(chunks, source=cld["url"], modality="audio", cloudinary_url=cld["url"])

    return transcript


# VIDEO
def ingest_video(video_path: str, max_frames: int = 10):
    """
    Ingest a local video:
      1. Upload to Cloudinary (video resource)
      2. Extract + transcribe audio → Qdrant
      3. Sample frames → CLIP embed → Qdrant
    """
    print(f"Ingesting video '{video_path}'")

    # 1. Cloudinary
    cld = upload_to_cloudinary(video_path, resource_type="video")
    print(f"   Cloudinary URL: {cld['url']}")

    with tempfile.TemporaryDirectory() as tmp:
        # 2. Audio → transcript
        audio_path = extract_audio_from_video(video_path, tmp)
        if audio_path:
            transcript = transcribe_audio(audio_path)
            chunks = chunk_text(transcript)
            upsert_text_chunks(
                chunks,
                source=f"{cld['url']}::audio",
                modality="video_audio",
                cloudinary_url=cld["url"],
            )
            print(f"   Audio: {len(chunks)} chunk(s)")
        else:
            print("   No audio track found")

        # 3. Frames → CLIP
        frames = extract_video_frames(video_path, max_frames=max_frames)
        print(f"   Frames: {len(frames)} extracted")
        for idx, frame in enumerate(tqdm(frames, desc="Embedding frames")):
            tags = suggest_tags(frame)
            upsert_image_vector(
                frame,
                source=f"{cld['url']}::frame_{idx}",
                tags=tags,
                cloudinary_url=cld["url"],
            )
            tag_text = f"Video frame {idx}. Tags: {', '.join(tags)}. Source: {cld['url']}"
            upsert_text_chunks(
                [tag_text],
                source=f"{cld['url']}::frame_{idx}",
                modality="video_frame",
                cloudinary_url=cld["url"],
            )

    print(f"   Video ingestion complete")

## Step 12 — Retrieval

In [ ]:
from qdrant_client.models import QueryRequest

def _collection_has_points(name: str) -> bool:
    try:
        info = qdrant_client.get_collection(name)
        return (info.points_count or 0) > 0
    except Exception:
        return False


def _search(collection: str, vector: list, top_k: int) -> list:
    """
    Compatibility wrapper: qdrant-client >= 1.7 removed .search() and
    replaced it with .query_points(). This wrapper tries the new API
    first and falls back to the old one so the notebook works on any version.
    """
    try:
        result = qdrant_client.query_points(
            collection_name=collection,
            query=vector,
            limit=top_k,
            with_payload=True,
        )
        return result.points
    except AttributeError:
        return qdrant_client.search(
            collection_name=collection,
            query_vector=vector,
            limit=top_k,
            with_payload=True,
        )


def retrieve_text_context(query: str, top_k: int = TOP_K) -> List[Dict]:
    if not _collection_has_points(TEXT_COLLECTION):
        return []
    q_emb = embed_texts([query])[0].tolist()
    hits  = _search(TEXT_COLLECTION, q_emb, top_k)
    return [
        {
            "text":                 h.payload.get("text", ""),
            "source":               h.payload.get("source", ""),
            "modality":             h.payload.get("modality", ""),
            "cloudinary_url":       h.payload.get("cloudinary_url", ""),
            "cloudinary_public_id": h.payload.get("cloudinary_public_id", ""),
            "score":                round(h.score, 4),
        }
        for h in hits
    ]


def retrieve_similar_images(query: str, top_k: int = 3) -> List[Dict]:
    if not _collection_has_points(IMAGE_COLLECTION):
        return []

    inputs = clip_processor(text=[query], return_tensors="pt", padding=True)
    with torch.no_grad():
        raw = clip_model.get_text_features(**inputs)
    t_feats = _to_tensor(raw)
    t_feats = t_feats / t_feats.norm(dim=-1, keepdim=True)
    q_emb   = t_feats.squeeze().cpu().numpy().tolist()

    hits = _search(IMAGE_COLLECTION, q_emb, top_k)
    return [
        {
            "source":               h.payload.get("source", ""),
            "caption":              h.payload.get("caption", ""),
            "tags":                 h.payload.get("tags", []),
            "cloudinary_url":       h.payload.get("cloudinary_url", ""),
            "cloudinary_public_id": h.payload.get("cloudinary_public_id", ""),
            "score":                round(h.score, 4),
        }
        for h in hits
    ]


print("Retrieval ready.")

## Step 13 — RAG Answer Generation (Groq LLaMA-3)

In [ ]:
SYSTEM_PROMPT = (
    "You are a precise AI assistant with access to a personal multimodal knowledge base.\n"
    "The knowledge base contains text notes, image descriptions, audio transcripts, and video content.\n"
    "Answer the user's question using ONLY the provided context snippets.\n"
    "If the context is insufficient, say so honestly — do not hallucinate.\n"
    "Cite the source index like [1] when you use a snippet.\n"
    "When an image or video is relevant, mention its Cloudinary URL so the user can view it."
)


def build_context_string(text_hits: List[Dict], image_hits: List[Dict]) -> str:
    parts = []
    for i, h in enumerate(text_hits, 1):
        cld = f" | url: {h['cloudinary_url']}" if h.get("cloudinary_url") else ""
        parts.append(
            f"[{i}] [{h['modality'].upper()}] score={h['score']}{cld}\n{h['text']}"
        )
    for j, h in enumerate(image_hits, len(text_hits) + 1):
        tag_str = ", ".join(h["tags"]) or "none"
        cld = f" | url: {h['cloudinary_url']}" if h.get("cloudinary_url") else ""
        parts.append(
            f"[{j}] [IMAGE] score={h['score']}{cld}\n"
            f"Caption: {h['caption']}\nTags: {tag_str}"
        )
    return "\n\n".join(parts) if parts else "No relevant context found in the knowledge base."


def ask(question: str, top_k: int = TOP_K, verbose: bool = False) -> str:
    """
    Full RAG pipeline:
      retrieve (text + images) → build context → Groq LLaMA-3 → answer
    """
    text_hits  = retrieve_text_context(question, top_k=top_k)
    image_hits = retrieve_similar_images(question, top_k=3)
    context    = build_context_string(text_hits, image_hits)

    if verbose:
        print("— Retrieved Context " + "─" * 40)
        print(textwrap.indent(context, "  "))
        print("—" * 60 + "\n")

    response = groq_client.chat.completions.create(
        model=GROQ_LLM_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": f"CONTEXT:\n{context}\n\nQUESTION: {question}"},
        ],
        temperature=0.2,
        max_tokens=1024,
    )
    return response.choices[0].message.content

---
## Step 14 — Ingest Your Data

Upload files to Colab first (Files panel → Upload), then run the relevant cell.

In [ ]:
# TEXT
sample_text = """
Retrieval-Augmented Generation (RAG) combines a retrieval system with a generative
language model. The retrieval step fetches relevant documents from a vector database,
and the generation step uses those documents as grounded context to produce accurate
answers. RAG reduces hallucinations and keeps responses up-to-date without retraining.
Common applications include enterprise Q&A systems, customer support bots, and personal
knowledge assistants.
"""
ingest_text(sample_text, source_name="rag_overview")

In [ ]:
# IMAGE
IMAGE_PATH = "/content/image.jpg"

if os.path.exists(IMAGE_PATH):
    tags = ingest_image(IMAGE_PATH)
    print(f"\nSuggested tags: {tags}")
else:
    print(f"Not found: {IMAGE_PATH} — upload the file first")

In [ ]:
# AUDIO
AUDIO_PATH = "/content/audio.mp3"

if os.path.exists(AUDIO_PATH):
    transcript = ingest_audio(AUDIO_PATH)
    print(f"\nFull transcript ({len(transcript)} chars) stored in Qdrant")
else:
    print(f"Not found: {AUDIO_PATH} — upload the file first")

In [ ]:
# VIDEO
VIDEO_PATH = "/content/video.mp4"

if os.path.exists(VIDEO_PATH):
    ingest_video(VIDEO_PATH, max_frames=10)
else:
    print(f"Not found: {VIDEO_PATH} — upload the file first")

---
## Step 15 — End-to-End Verification Suite

Run all four cells below **after** completing Step 14.  
Each cell tests one modality, shows what was retrieved from Qdrant (including the  
Cloudinary URL stored in the payload), and prints the grounded RAG answer.

A  verdict confirms whether the answer is actually grounded in ingested content.


In [ ]:


def _verdict(answer: str, hits: list) -> str:
    """Return  if the LLM used at least one retrieved hit, else ."""
    if not hits:
        return " No hits retrieved — nothing was ingested for this modality yet."
    answer_lower = answer.lower()
    for h in hits:
        text = h.get("text") or h.get("caption") or " ".join(h.get("tags", []))
        if any(word in answer_lower for word in text.lower().split() if len(word) > 4):
            return "  Answer is grounded in retrieved context."
    return "Answer generated but overlap with context is low — check ingestion."

def verify(question: str, modality_filter: str = None, label: str = ""):
    """
    Full RAG pipeline with per-hit display and a grounding verdict.
    modality_filter: if set, only show hits with matching payload modality.
    """
    print(f"\n{'═'*65}")
    print(f"  {label or question}")
    print(f"{'═'*65}")

    text_hits  = retrieve_text_context(question, top_k=TOP_K)
    image_hits = retrieve_similar_images(question, top_k=3)


    display_text  = [h for h in text_hits  if not modality_filter or h["modality"] == modality_filter]
    display_image = image_hits if (not modality_filter or modality_filter == "image") else []

    print(f"\n   Retrieved hits ({len(display_text)} text, {len(display_image)} image):")
    for i, h in enumerate(display_text, 1):
        cld = h.get("cloudinary_url") or "(no URL — inline text)"
        print(f"    [{i}] modality={h['modality']}  score={h['score']}")
        print(f"         cloudinary_url: {cld}")
        print(f"         text preview  : {h['text'][:120].strip()}...")
    for j, h in enumerate(display_image, len(display_text)+1):
        print(f"    [{j}] IMAGE  score={h['score']}")
        print(f"         cloudinary_url: {h.get('cloudinary_url','')}")
        print(f"         tags          : {', '.join(h.get('tags',[]))}")

    context = build_context_string(text_hits, image_hits)
    answer  = groq_client.chat.completions.create(
        model=GROQ_LLM_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": f"CONTEXT:\n{context}\n\nQUESTION: {question}"},
        ],
        temperature=0.2,
        max_tokens=512,
    ).choices[0].message.content

    all_hits = display_text + display_image
    print(f"\n   Answer:\n")
    print(textwrap.indent(answer, "     "))
    print(f"\n  {_verdict(answer, all_hits)}")
    print(f"{'─'*65}")
    return answer

print("Verification helpers loaded — run the four modality cells below.")


In [ ]:

verify(
    question        = "What is Retrieval-Augmented Generation and why is it useful?",
    modality_filter = "text",
    label           = "TEXT    What is RAG and why is it useful?",
)


In [ ]:

verify(
    question        = "What images are stored in the knowledge base? Describe them and show the Cloudinary URLs.",
    modality_filter = "image",
    label           = "IMAGE    What images are in the knowledge base?",
)


In [ ]:

verify(
    question        = "What was said in the audio recording? Summarise the key points.",
    modality_filter = "audio",
    label           = "AUDIO    What was said in the audio file?",
)


In [ ]:

    question        = "What happened in the video? Describe the scenes and any speech.",
    modality_filter = "video_audio",
    label           = "VIDEO (audio track)    What happened in the video?",
)
verify(
    question        = "What visual content was captured in the video frames?",
    modality_filter = "video_frame",
    label           = "VIDEO (frames)    What did the video look like?",
)


In [ ]:

print("\n" + "═"*65)
print("  CROSS-MODAL    Everything stored in the knowledge base")
print("="*65)

all_text  = retrieve_text_context("knowledge base content summary", top_k=20)
all_image = retrieve_similar_images("content summary", top_k=5)

print(f"\n  Total text  hits : {len(all_text)}")
print(f"  Total image hits : {len(all_image)}")

modality_counts = {}
for h in all_text:
    m = h.get("modality","unknown")
    modality_counts[m] = modality_counts.get(m, 0) + 1
print("\n  Breakdown by modality:")
for m, cnt in sorted(modality_counts.items()):
    print(f"    • {m:15s} : {cnt} chunk(s)")

print("\n  Cloudinary URLs stored:")
seen = set()
for h in all_text:
    url = h.get("cloudinary_url","")
    if url and url not in seen:
        seen.add(url)
        print(f"    → {url}")
for h in all_image:
    url = h.get("cloudinary_url","")
    if url and url not in seen:
        seen.add(url)
        print(f"    → {url}")
if not seen:
    print("    (none — only inline text has been ingested so far)")

verify(
    question = "Give me a complete summary of everything stored: text notes, images, audio, and video.",
    label    = "CROSS-MODAL    Full knowledge base summary",
)


---
## Step 16 — Standalone Image Tag Suggester

In [ ]:
def get_tags_for_image(image_path: str, top_k: int = 10) -> List[str]:
    """Return CLIP zero-shot tags for any image — without ingesting it."""
    img = Image.open(image_path).convert("RGB")
    return suggest_tags(img, top_k=top_k, threshold=0.15)

TAG_PATH = "/content/image.jpg"

if os.path.exists(TAG_PATH):
    tags = get_tags_for_image(TAG_PATH)
    print(f"Tags for '{TAG_PATH}':")
    for t in tags:
        print(f"   • {t}")
else:
    print(f"Upload the image first: {TAG_PATH}")

---
## Step 17 — Interactive Chat Loop

In [ ]:
print("Multimodal RAG Chat — type 'quit' to exit\n")

while True:
    try:
        q = input("You: ").strip()
    except (EOFError, KeyboardInterrupt):
        print("\nGoodbye!")
        break
    if not q:
        continue
    if q.lower() in ("quit", "exit", "bye", "q"):
        print("Goodbye!")
        break
    answer = ask(q)
    print(f"\nBot: {answer}\n{'─'*55}\n")

Multimodal RAG Chat — type 'quit' to exit



---
## Step 18 — Utilities

In [ ]:
# Collection stats
for col in [TEXT_COLLECTION, IMAGE_COLLECTION]:
    info = qdrant_client.get_collection(col)
    print(f"'{col}': {info.points_count} points")